In [0]:
from pyspark.sql.functions import col,trim

bronze_df = spark.read.table("bronze_universities")
# 2. Apply Silver layer cleaning transformations
silver_universities_df =(
    bronze_df
    # Remove duplicate rows based on university_id
    .dropDuplicates(["university_id"])
    # Clean up accidental whitespace in string columns
    .withColumn("university_name",trim(col("university_name")))
    .withColumn("country",trim(col("country")))
    .withColumn("city",trim(col("city")))
    # Ensure critical columns do not contain null values
    .filter(col("university_id").isNotNull() & col("university_name").isNotNull())
)
# 3. Save the cleaned DataFrame as a managed Silver Delta table
(
    silver_universities_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_universities")
)
print("Silver table successfully created and saved: silver_universities")
verified_silver_df = spark.read.table("silver_universities")
verified_silver_df.show(10,truncate=False)
print("Total records in Silver table:", verified_silver_df.count())